## Implementation of the BB84 protocol

This notebook implements the **BB84 quantum key distribution (QKD) protocol** in a fully classical simulation, using Python objects to represent qubits and their measurement statistics. This illustrates the logical structure of BB84 and how an eavesdropper (Eve) inevitably introduces errors that Alice and Bob can detect.

The implementation is split into four steps:

1. Alice prepares random classical bits and encodes them in randomly chosen qubit bases (rectilinear or diagional).
2. Bob (and optionally Eve) measures the qubits in randomly chosen bases.
3. Alice and Bob perform **sifting**: they compare bases over a classical channel and discard all measurements where their bases differed.
4. They estimate the **error rate** on the sifted key. If it is too high, they abort; otherwise, they can distil a secret key.

All quantum behavior (basis dependence, random outcomes when bases mismatch) is implemented in `qubits.py` and `measurement.py`.

In [17]:
from qubits import XQubit, ZQubit, NQubitChain
from measurement import measure_nqubit_chain, error_rate
import random

### (1) Alice chooses a string of *N* random classical bits and encodes these in a random sequence of *N* qubit polarisation states

In [18]:
N = 20
alice_bitstring = random.choices([0, 1], k=N)
alice_polars = random.choices(["Z", "X"], k=N)

qchain = NQubitChain(alice_bitstring, alice_polars)

In [19]:
print("Alice's bit string:", "".join(str(b) for b in alice_bitstring))
print("Alice's bases:", alice_polars)

Alice's bit string: 11111101101100101101
Alice's bases: ['Z', 'Z', 'X', 'Z', 'Z', 'Z', 'X', 'Z', 'Z', 'X', 'Z', 'X', 'X', 'X', 'Z', 'Z', 'Z', 'X', 'X', 'X']


### (2) Bob and (optionally) Eve measure the qubits in random bases

In [20]:
eavesdrop = True
if eavesdrop:
    eve_polars = random.choices(["Z", "X"], k=N)
    eve_bitstring = measure_nqubit_chain(qchain, eve_polars)

In [21]:
bob_polars = random.choices(["Z", "X"], k=N)
bob_bitstring = measure_nqubit_chain(qchain, bob_polars)

In [22]:
print("Bob's bit string:", "".join(str(b) for b in bob_bitstring))
print("Bob's bases:", bob_polars)

Bob's bit string: 11111000000100101101
Bob's bases: ['Z', 'X', 'X', 'Z', 'X', 'X', 'Z', 'X', 'Z', 'Z', 'Z', 'Z', 'X', 'Z', 'Z', 'Z', 'X', 'X', 'X', 'Z']


Classical post processing

### (3) Bob and Alice perform sifting by comparing bases they measured and prepared qubits in. They discard any qubit measurements from different bases

In [ ]:
from typing import Literal
from collections.abc import Sequence

def remove_different_basis(
    bitstring: Sequence[Literal[0, 1]], basis_mask: Sequence[Literal[True, False]]
):
    return [bit for bit, keep in zip(bitstring, basis_mask) if keep]

In [24]:
common_bases = [a == b for a, b in zip(alice_polars, bob_polars)]

bob_remaining_bits = remove_different_basis(bob_bitstring, common_bases)
alice_remaining_bits = remove_different_basis(alice_bitstring, common_bases)

In [29]:
print("Bob's remaining bits:", "".join(str(b) for b in bob_remaining_bits))
print("Alice's remaining bits:", "".join(str(b) for b in alice_remaining_bits))

Bob's remaining bits: 1110001010
Alice's remaining bits: 1111101010


### (4) Parameter estimation step: determine the error rate of the channel

In [27]:
eps = 0
if error_rate(alice_remaining_bits, bob_remaining_bits) > eps:
    print("Secret Key could not be established")
else:
    key_str = "".join(str(b) for b in alice_remaining_bits)
    print("Key established. Key:", key_str)

Secret Key could not be established
